<a href="https://colab.research.google.com/github/ubuntumel/AI_Colab_DeepLearning/blob/main/Cifar10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://colab.research.google.com/drive/1-TA6KDdvBS4xCRqLqb0oXxHtm54t2MGp?usp=sharing

Authors: Meliton Rojas, Bricio Blancas Salgado

In [ ]:
# import cifar10 dataset as well as the models and layers.
import keras
from keras import models
from keras import layers
from keras import optimizers
from keras import activations
from keras.datasets import cifar10

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

This dataset contains 50,000 32 * 32 color training images and 10,000 test images labeled over 10 categories. The inputs are 32 * 32 color images as well as the 3 RGB color channels. The 10 labels are integers 0-9 [airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck]. The dimensions of this dataset is 60,000 images split 50,000 for training and 10,000 for testing.

In [ ]:
# load training and testing data from cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
# Reshape images from (32, 32, 3) to flattened vector of (3072)
# convert to float
x_train = x_train.reshape((x_train.shape[0], 32 * 32 * 3)).astype("float32") / 255.0
x_test  = x_test.reshape((x_test.shape[0],  32 * 32 * 3)).astype("float32") / 255.0

In [ ]:
# build_model function creates a neural network
# Parameters:
# hidden_units: number of neurons per hidden layer
# num_hidden_layers: how many hidden layers
# activation: activation function for hidden layers
# num_classes: number of output classes

def build_model(input_dim = 32 * 32 * 3, hidden_units = 64, num_hidden_layers = 1, activation = "relu", num_classes = 10) -> keras.Model:
  # Create the Sequential model
  model = keras.Sequential(name=f"model_{activation}_{hidden_units}_{num_hidden_layers}h1")

  # Input layer
  model.add(layers.Input(shape=(input_dim,)))

  # add hidden layers dynamically
  for _ in range(num_hidden_layers):
    model.add(layers.Dense(hidden_units, activation=activation))

  # output layer with softmax for multi classification
  model.add(layers.Dense(num_classes, activation="softmax"))

  return model

In [ ]:
# fit_evaluate training and evaluation function
# compiles, trains, and evaluates model
# returns training history, test loss, and test accuracy

def fit_evaluate(model: keras.Model,
                 x_train, y_train,
                 x_test, y_test,
                 optimizer = "rmsprop",
                 loss = "sparse_categorical_crossentropy",
                 epochs = 20,
                 batch_size = 128,
                 validation_split = 0.1,
                 verbose = 2, metrics = None):

  # automatically choose correct accuracy metric if not specified
    if metrics is None:
      if loss == "categorical_crossentropy":
          metrics = ["categorical_accuracy"]
      else:
          metrics = ["sparse_categorical_accuracy"]
    # compile the model
    model.compile(
        optimizer = optimizer,
        loss = loss,
        metrics = ["accuracy"],
    )
    # train the model
    history = model.fit(
        x_train, y_train,
        epochs = epochs,
        batch_size = batch_size,
        validation_split = validation_split,
        verbose = verbose
    )
    # evaluate test data
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose = 0)

    return history, test_loss, test_acc

In [ ]:
# Train 3 models with different activations.
activations_list = ["relu", "tanh", "sigmoid"]
results = []

# train and evaluate each activation function
for activation in activations_list:
  print(f"Training model: {activation}")
  model = build_model(hidden_units = 64, num_hidden_layers = 1, activation = activation)
  history, test_loss, test_acc = fit_evaluate(model, x_train, y_train, x_test, y_test, optimizer="rmsprop", loss = "sparse_categorical_crossentropy", epochs = 20, batch_size = 128, verbose = 2)
  # store results
  results.append({"Activation": activation, "Test Accuracy": float(test_acc)})
#display results sorted by accuracy
df = pd.DataFrame(results).sort_values("Test Accuracy", ascending = False)
display(df)

Training model: relu
Epoch 1/20
352/352 - 4s - 12ms/step - accuracy: 0.2242 - loss: 2.1041 - val_accuracy: 0.3112 - val_loss: 1.9659
Epoch 2/20
352/352 - 4s - 10ms/step - accuracy: 0.3158 - loss: 1.9004 - val_accuracy: 0.3158 - val_loss: 1.8775
Epoch 3/20
352/352 - 3s - 8ms/step - accuracy: 0.3413 - loss: 1.8401 - val_accuracy: 0.3498 - val_loss: 1.8012
Epoch 4/20
352/352 - 4s - 11ms/step - accuracy: 0.3551 - loss: 1.7995 - val_accuracy: 0.3480 - val_loss: 1.8059
Epoch 5/20
352/352 - 3s - 8ms/step - accuracy: 0.3680 - loss: 1.7665 - val_accuracy: 0.3734 - val_loss: 1.7826
Epoch 6/20
352/352 - 3s - 8ms/step - accuracy: 0.3755 - loss: 1.7468 - val_accuracy: 0.3620 - val_loss: 1.7840
Epoch 7/20
352/352 - 3s - 8ms/step - accuracy: 0.3846 - loss: 1.7277 - val_accuracy: 0.3702 - val_loss: 1.7582
Epoch 8/20
352/352 - 4s - 11ms/step - accuracy: 0.3902 - loss: 1.7124 - val_accuracy: 0.3106 - val_loss: 2.0007
Epoch 9/20
352/352 - 3s - 8ms/step - accuracy: 0.3936 - loss: 1.6972 - val_accuracy: 0.

,Activation,Test Accuracy
2,sigmoid,0.4533
1,tanh,0.4148
0,relu,0.4134


In [ ]:
# Identify best activation function based on test accuracy
best_activation = df.iloc[0]["Activation"]
best_accuracy = df.iloc[0]["Test Accuracy"]
print(f"\nBest activation: {best_activation} (test accuracy = {best_accuracy:.4f})")


Best activation: sigmoid (test accuracy = 0.4533)


In [ ]:
# Model tuning testing different hidden layer sizes 128, 256, 512
tuning_results = []

for units in [128, 256, 512]:
  print(f"Tuning: units = {units}, layers = 1, activation = {best_activation}")
  model = build_model(hidden_units = units, num_hidden_layers = 1, activation = best_activation)

  history, test_loss, test_acc = fit_evaluate(model, x_train, y_train, x_test, y_test, optimizer="rmsprop", loss = "sparse_categorical_crossentropy", epochs = 20, batch_size = 128, verbose = 2)
  tuning_results.append({"Stage": "Units Sweeps (1 layer)", "Activation": best_activation, "Hidden Units": units, "# Hidden Layers": 1, "Optimizer": "rmsprop", "Loss": "sparse_categorical_crossentropy", "Test Accuracy": float(test_acc)})

Tuning: units = 128, layers = 1, activation = sigmoid
Epoch 1/20
352/352 - 8s - 22ms/step - accuracy: 0.2933 - loss: 1.9631 - val_accuracy: 0.3040 - val_loss: 1.8889
Epoch 2/20
352/352 - 4s - 10ms/step - accuracy: 0.3631 - loss: 1.7889 - val_accuracy: 0.3602 - val_loss: 1.8033
Epoch 3/20
352/352 - 5s - 14ms/step - accuracy: 0.3928 - loss: 1.7140 - val_accuracy: 0.3796 - val_loss: 1.7293
Epoch 4/20
352/352 - 4s - 10ms/step - accuracy: 0.4108 - loss: 1.6604 - val_accuracy: 0.3962 - val_loss: 1.7129
Epoch 5/20
352/352 - 5s - 15ms/step - accuracy: 0.4260 - loss: 1.6207 - val_accuracy: 0.3998 - val_loss: 1.6755
Epoch 6/20
352/352 - 5s - 14ms/step - accuracy: 0.4386 - loss: 1.5866 - val_accuracy: 0.4214 - val_loss: 1.6340
Epoch 7/20
352/352 - 4s - 10ms/step - accuracy: 0.4453 - loss: 1.5609 - val_accuracy: 0.4434 - val_loss: 1.5764
Epoch 8/20
352/352 - 5s - 15ms/step - accuracy: 0.4566 - loss: 1.5345 - val_accuracy: 0.4508 - val_loss: 1.5555
Epoch 9/20
352/352 - 5s - 13ms/step - accuracy: 0.

In [ ]:
# determine the units
df_units = pd.DataFrame([r for r in tuning_results if r["Stage"] == "Units Sweeps (1 layer)"])
best_units = int(df_units.sort_values("Test Accuracy", ascending=False).iloc[0]["Hidden Units"])
print(f"\nBest hidden units: {best_units}")


Best hidden units: 512


In [ ]:
# add hidden layer
print(f"Tuning: units = {best_units}, layers = 2, activation = {best_activation}")
model_extra = build_model(hidden_units=best_units, num_hidden_layers=2, activation=best_activation)

hist_extra, loss_extra, acc_extra = fit_evaluate(
    model_extra,
    x_train, y_train,
    x_test, y_test,
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    epochs=20,
    batch_size=128,
    verbose=2
)

tuning_results.append({
    "Stage": "Add hidden layer",
    "Activation": best_activation,
    "Hidden Units": best_units,
    "# Hidden Layers": 2,
    "Optimizer": "rmsprop",
    "Loss": "sparse_categorical_crossentropy",
    "Test Accuracy": float(acc_extra)
})

# changing optimizer
print(f"Tuning: Adam optimizer (units={best_units}, layers=2, act={best_activation})")
model_adam = build_model(hidden_units=best_units, num_hidden_layers=2, activation=best_activation)

hist_adam, loss_adam, acc_adam = fit_evaluate(
    model_adam,
    x_train, y_train,
    x_test, y_test,
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    epochs=20,
    batch_size=128,
    verbose=2
)

tuning_results.append({
    "Stage": "Change optimizer",
    "Activation": best_activation,
    "Hidden Units": best_units,
    "# Hidden Layers": 2,
    "Optimizer": "adam",
    "Loss": "sparse_categorical_crossentropy",
    "Test Accuracy": float(acc_adam)
})

Tuning: units = 512, layers = 2, activation = sigmoid
Epoch 1/20
352/352 - 14s - 40ms/step - accuracy: 0.2615 - loss: 2.0194 - val_accuracy: 0.2426 - val_loss: 2.1693
Epoch 2/20
352/352 - 13s - 38ms/step - accuracy: 0.3479 - loss: 1.8071 - val_accuracy: 0.3746 - val_loss: 1.7583
Epoch 3/20
352/352 - 20s - 58ms/step - accuracy: 0.3879 - loss: 1.7107 - val_accuracy: 0.4004 - val_loss: 1.6776
Epoch 4/20
352/352 - 13s - 38ms/step - accuracy: 0.4136 - loss: 1.6454 - val_accuracy: 0.3972 - val_loss: 1.6568
Epoch 5/20
352/352 - 13s - 38ms/step - accuracy: 0.4296 - loss: 1.5965 - val_accuracy: 0.4388 - val_loss: 1.5797
Epoch 6/20
352/352 - 13s - 38ms/step - accuracy: 0.4433 - loss: 1.5544 - val_accuracy: 0.3952 - val_loss: 1.6590
Epoch 7/20
352/352 - 15s - 42ms/step - accuracy: 0.4548 - loss: 1.5231 - val_accuracy: 0.4296 - val_loss: 1.5688
Epoch 8/20
352/352 - 13s - 36ms/step - accuracy: 0.4674 - loss: 1.4949 - val_accuracy: 0.4700 - val_loss: 1.4798
Epoch 9/20
352/352 - 13s - 37ms/step - acc

In [ ]:
# Switch labels to one-hot and use categorical_crossentropy
# Change loss function
from keras.utils import to_categorical
y_train_oh = to_categorical(y_train, 10)
y_test_oh  = to_categorical(y_test, 10)


print(f"Tuning: categorical_crossentropy (one-hot) + Adam (units = {best_units}, layers = 2, activation = {best_activation})")
model_cce = build_model(hidden_units=best_units, num_hidden_layers=2, activation=best_activation)

hist_cce, loss_cce, acc_cce = fit_evaluate(
    model_cce,
    x_train, y_train_oh,
    x_test, y_test_oh,
    optimizer="adam",
    loss="categorical_crossentropy",
    epochs=20,
    batch_size=128,
    verbose=2
)

tuning_results.append({
    "Stage": "Change loss (one-hot)",
    "Activation": best_activation,
    "Hidden Units": best_units,
    "# Hidden Layers": 2,
    "Optimizer": "adam",
    "Loss": "categorical_crossentropy",
    "Test Accuracy": float(acc_cce)
})

# display tuning results and what seems to be the best
df_tune = pd.DataFrame(tuning_results).sort_values("Test Accuracy", ascending=False)
print("\nResults:")
display(df_tune)


best_step5 = df_tune.iloc[0]
print("\nBest tuned configuration:")
display(df_tune.head(1))

Tuning: categorical_crossentropy (one-hot) + Adam (units = 512, layers = 2, activation = sigmoid)
Epoch 1/20
352/352 - 20s - 57ms/step - accuracy: 0.3072 - loss: 1.9131 - val_accuracy: 0.3640 - val_loss: 1.7975
Epoch 2/20
352/352 - 18s - 50ms/step - accuracy: 0.3913 - loss: 1.6993 - val_accuracy: 0.4162 - val_loss: 1.6592
Epoch 3/20
352/352 - 17s - 48ms/step - accuracy: 0.4199 - loss: 1.6233 - val_accuracy: 0.4216 - val_loss: 1.6377
Epoch 4/20
352/352 - 19s - 54ms/step - accuracy: 0.4393 - loss: 1.5651 - val_accuracy: 0.4458 - val_loss: 1.5700
Epoch 5/20
352/352 - 17s - 48ms/step - accuracy: 0.4536 - loss: 1.5343 - val_accuracy: 0.4218 - val_loss: 1.6181
Epoch 6/20
352/352 - 22s - 63ms/step - accuracy: 0.4620 - loss: 1.5050 - val_accuracy: 0.4568 - val_loss: 1.5197
Epoch 7/20
352/352 - 16s - 45ms/step - accuracy: 0.4752 - loss: 1.4683 - val_accuracy: 0.4614 - val_loss: 1.5093
Epoch 8/20
352/352 - 17s - 49ms/step - accuracy: 0.4830 - loss: 1.4504 - val_accuracy: 0.4656 - val_loss: 1.484

,Stage,Activation,Hidden Units,# Hidden Layers,Optimizer,Loss,Test Accuracy
4,Change optimizer,sigmoid,512,2,adam,sparse_categorical_crossentropy,0.5125
5,Change loss (one-hot),sigmoid,512,2,adam,categorical_crossentropy,0.5036
2,Units Sweeps (1 layer),sigmoid,512,1,rmsprop,sparse_categorical_crossentropy,0.4904
1,Units Sweeps (1 layer),sigmoid,256,1,rmsprop,sparse_categorical_crossentropy,0.4871
3,Add hidden layer,sigmoid,512,2,rmsprop,sparse_categorical_crossentropy,0.4774
0,Units Sweeps (1 layer),sigmoid,128,1,rmsprop,sparse_categorical_crossentropy,0.4511



Best tuned configuration:


,Stage,Activation,Hidden Units,# Hidden Layers,Optimizer,Loss,Test Accuracy
4,Change optimizer,sigmoid,512,2,adam,sparse_categorical_crossentropy,0.5125


We chose the ReLU activation function because it is one of the most common activation functions in neural networks. ReLu outputs zero for negative values and the input value for positive values. Helps reduce the vanishing gradient problem and allows the model to train faster and efficiently. ReLU is computationally simple and performs well in deep learning. Tanh was chosen because it outputs values beween [-1, 1], making it zero centered. Zero centered can help improve optimization compared to sigmoid because gradients may balance more effectively during training. Tanh can capture nonlinear relationships than sigmoid. Sigmoid was chosen because it outputs values between 0 and 1. It provided a useful comparison to the other activation functions.

The sigmoid activation function seemed to achieve the highest test accuracy with (0.4533) outperforming tanh (0.4148) and ReLu (0.4314). Sigmoid performed better for this specific dataset configuration. May be due to the shallow structure of the model. The neural network was not very deep, the vanishing gradient issue associated with sigmoid was not visible on this performance.

Tuning the model using sigmoid activation function and increasing the number of neurons in the hidden layers by 128, 256, and 512 increased the models capacity to learn more patterns. The larger hidden layers improved performance but began to level off as the number of neurons increased.

Adding hidden layers allowed the model to capture nonlinear relationships. The deeper model improved performance compared to the single hidden layer.

Changing the optimizer from RMSprop to Adam futher improved stability and test accuracy. Adam's adaptive learning rate lead to efficient training.

Sparse categorical crossentropy to categorical crossentropy did not really change much in the performance. Both loss functions are essentially the same.

Tuning the model improved performance over the orignal models. The best results were achieved with sigmoid activation functions with increased hidden units, layers and optimizer. Maybe things would be different with deeper networks as well as epochs.